In [1]:
# Use the native inference API to send a text message to Amazon Titan Text G1 - Express.

import boto3
import json

from botocore.exceptions import ClientError

# Create an Amazon Bedrock Runtime client.
brt = boto3.client("bedrock-runtime")

# Set the model ID, e.g., Amazon Titan Text G1 - Express.
model_id = "anthropic.claude-3-5-sonnet-20241022-v2:0"

# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Format the request payload using the model's native structure.
native_request = {
    "inputText": prompt,
    "textGenerationConfig": {
        "maxTokenCount": 512,
        "temperature": 0.5,
        "topP": 0.9
    },
}

# native_request = {
#   "modelId": "anthropic.claude-3-5-haiku-20241022-v1:0",
#   "contentType": "application/json",
#   "accept": "application/json",
#   "body": {
#     "anthropic_version": "bedrock-2023-05-31",
#     "max_tokens": 200,
#     "top_k": 250,
#     "stopSequences": [],
#     "temperature": 1,
#     "top_p": 0.999,
#     "messages": [
#       {
#         "role": "user",
#         "content": [
#           {
#             "type": "text",
#             "text": prompt
#           }
#         ]
#       }
#     ]
#   }
# }

# Convert the native request to JSON.
request = json.dumps(native_request)

try:
    # Invoke the model with the request.
    response = brt.invoke_model(modelId=model_id, body=request)

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
    exit(1)

# Decode the response body.
model_response = json.loads(response["body"].read())

# Extract and print the response text.
response_text = model_response["results"][0]["outputText"]
print(response_text)

ERROR: Can't invoke 'anthropic.claude-3-5-sonnet-20241022-v2:0'. Reason: An error occurred (ValidationException) when calling the InvokeModel operation: Invocation of model ID anthropic.claude-3-5-sonnet-20241022-v2:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.


NameError: name 'response' is not defined

In [12]:
rp = '''
You are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.

    Task Overview:
    Generate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.

    Guidelines for Generating Q&A Pairs:
    1. **Question Design**:
    - Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.
    - Avoid single-hop or trivial questions directly derived from a single triple.
    - The answer should be an entity or node in the graph.

    2. **Multi-Hop Reasoning**:
    - Use paths connecting entities indirectly through multiple relationships to infer answers.
    - Questions should reflect meaningful and interesting connections within the graph.
    - Aim for question with at least 4 hops or higher whenever possible.

    3. **Answer Validation**:
    - Ensure each answer is fully supported by one or more triples from the graph.
    - Include the exact path (a sequence of triples) that justifies the answer.

    4. **Comprehensive Coverage**:
    - Generate as many high-quality Q&A pairs as possible, exploring all meaningful paths and connections within the graph.

    5. **Fallback Condition**:
    - If no valid Q&A pairs can be generated from the graph, explicitly indicate this in the response.

    Graph Data:
    Below is the graph data for your task:
    [['Beetle', 'parentTaxon', 'Holometabola_Q37140800'], ['Golden_swallow', 'consumes', 'Insect'], ['Golden_swallow', 'consumes', 'Hemiptera'], ['Earwig', 'parentTaxon', 'Insect'], ['Earwig', 'consumes', 'Aphid'], ['Great_spotted_woodpecker', 'consumes', 'Beetle'], ['Great_spotted_woodpecker', 'consumes', 'Aphid'], ['Green_pheasant', 'consumes', 'Insect'], ['Green_pheasant', 'consumes', 'Plant'], ['Fly', 'parentTaxon', 'Holometabola_Q37140800'], ['European_bee-eater', 'consumes', 'Beetle'], ['European_bee-eater', 'consumes', 'Bee']]
    
    Repond ONLY with JSON with the following structure:

    - **valid_qa_pairs**: Boolean indicating if valid QA pairs were generated.
    - **number_of_qa_pairs**: Integer specifying the total number of QA pairs.
    - **qa_pairs**: A list of QA pairs, where each pair includes:
    - **question**: String representing the question.
    - **answer**: String representing the answer.
    - **supporting_path**: A list of triples, where each triple includes:
        - **subject**: String representing the subject.
        - **predicate**: String representing the predicate.
        - **object**: String representing the object.
'''

In [14]:
# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: Apache-2.0
"""
Example of building a JSON payload for Anthropic Claude and invoking AWS Bedrock
with retries and error handling.
"""
import boto3
import json
import logging
import time

from botocore.exceptions import ClientError

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def build_anthropic_request_body(
    system_prompt: str,
    user_prompt: str,
    max_tokens: int,
    temperature: float = 1.0
) -> dict:
    """
    Builds the JSON payload for Anthropic Claude.

    :param system_prompt: Instructions or context for the system.
    :param user_prompt: The text of the user's request.
    :param max_tokens: The maximum number of tokens to generate.
    :param temperature: Sampling temperature (creativity control).
    :return: A dict representing the request body for Claude.
    """
    # For Anthropic on Bedrock, the required fields typically include:
    # - anthropic_version
    # - system (system instructions)
    # - messages (the conversation so far)
    # - max_tokens (or max_tokens_to_sample in older versions)
    # - Optionally: temperature, top_p, etc.

    # Note: If your particular Claude model expects a single combined prompt,
    # you can instead pass a single string in "prompt". This example uses
    # "system" and "messages" to reflect the multi-message format.
    

    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "system": system_prompt,
        "messages": [
            {"role": "user", "content": user_prompt}
        ],
        "max_tokens": max_tokens,
        "temperature": temperature
    }

    return request_body


def invoke_bedrock_endpoint(
    request_body: dict,
    model_id: str,
    region_name: str = "us-east-1",
    max_retries: int = 3,
    backoff_factor: float = 2.0
) -> dict:
    """
    Invokes the Bedrock endpoint with a given request body and model ID,
    with exponential backoff retries for transient errors.

    :param request_body: JSON payload specific to the chosen model.
    :param model_id: The Bedrock model ID, e.g. "anthropic.claude-v1".
    :param region_name: The AWS region to call. Default is "us-east-1".
    :param max_retries: Number of retry attempts for transient errors.
    :param backoff_factor: Factor for exponential backoff, e.g. 2.0 means
                           1s, 2s, 4s between retries, etc.
    :return: The deserialized JSON response from Bedrock.
    """
    bedrock_runtime = boto3.client(
        service_name='bedrock-runtime',
        region_name=region_name
    )

    for attempt in range(max_retries):
        try:
            # Serialize the request body as JSON.
            body_json = json.dumps(request_body)

            response = bedrock_runtime.invoke_model(
                body=body_json,
                modelId=model_id,
                contentType='application/json'
            )

            # The response body is a StreamingBody, so we need to read and decode it.
            response_body = json.loads(response.get('body').read())
            return response_body

        except ClientError as err:
            logger.error(
                "Error invoking Bedrock on attempt %s: %s",
                attempt + 1,
                err.response["Error"]["Message"]
            )

            # If this was the last attempt, re-raise the error.
            if attempt == max_retries - 1:
                raise

            # Otherwise, back off exponentially before retrying.
            sleep_time = backoff_factor ** attempt
            logger.info(f"Retrying in {sleep_time} seconds...")
            time.sleep(sleep_time)


# Example usage (one single call):
if __name__ == "__main__":
    try:
        # Build the payload for Anthropic Claude.
        request_json = build_anthropic_request_body(
            system_prompt="You are a useful assistant.",
            user_prompt=rp,
            max_tokens=1024,
            temperature=0
        )

        # Model ID for your chosen Anthropic Claude variant on Bedrock.
        model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"

        # Invoke the endpoint.
        response_data = invoke_bedrock_endpoint(request_json, model_id)
        print("Response from Claude:", json.dumps(response_data, indent=2))

    except ClientError as e:
        logger.error(f"Failed after retries: {e}")


Response from Claude: {
  "id": "msg_bdrk_01NV4byMS3ipPY6RRWGyqtv4",
  "type": "message",
  "role": "assistant",
  "model": "claude-3-5-sonnet-20241022",
  "content": [
    {
      "type": "text",
      "text": "{\n    \"valid_qa_pairs\": true,\n    \"number_of_qa_pairs\": 3,\n    \"qa_pairs\": [\n        {\n            \"question\": \"Which bird species can eat both insects that are classified under Holometabola?\",\n            \"answer\": \"European_bee-eater\",\n            \"supporting_path\": [\n                {\n                    \"subject\": \"Beetle\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Holometabola_Q37140800\"\n                },\n                {\n                    \"subject\": \"Fly\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Holometabola_Q37140800\"\n                },\n                {\n                    \"subject\": \"European_bee-eater\",\n                    \"p

In [23]:
response_data['content'][0]['text']
a = json.loads(response_data['content'][0]['text'])

In [25]:
a

{'valid_qa_pairs': True,
 'number_of_qa_pairs': 3,
 'qa_pairs': [{'question': 'Which bird species can eat both insects that are classified under Holometabola?',
   'answer': 'European_bee-eater',
   'supporting_path': [{'subject': 'Beetle',
     'predicate': 'parentTaxon',
     'object': 'Holometabola_Q37140800'},
    {'subject': 'Fly',
     'predicate': 'parentTaxon',
     'object': 'Holometabola_Q37140800'},
    {'subject': 'European_bee-eater',
     'predicate': 'consumes',
     'object': 'Beetle'}]},
  {'question': 'Which birds can eat the same prey as Earwigs?',
   'answer': 'Great_spotted_woodpecker',
   'supporting_path': [{'subject': 'Earwig',
     'predicate': 'consumes',
     'object': 'Aphid'},
    {'subject': 'Great_spotted_woodpecker',
     'predicate': 'consumes',
     'object': 'Aphid'}]},
  {'question': "Which birds can feed on both insects that belong to Holometabola and those that don't?",
   'answer': 'Great_spotted_woodpecker',
   'supporting_path': [{'subject': 'Be

In [2]:
# Use the ListFoundationModels API to show the models that are available in your region.
import boto3
             
# Create an &BR; client in the &region-us-east-1; Region.
bedrock = boto3.client(
    service_name="bedrock"
)

bedrock.list_foundation_models()

{'ResponseMetadata': {'RequestId': '7cd5b98f-cbc4-4243-9262-cfe31ee0abaf',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 13 Jan 2025 07:05:45 GMT',
   'content-type': 'application/json',
   'content-length': '33511',
   'connection': 'keep-alive',
   'x-amzn-requestid': '7cd5b98f-cbc4-4243-9262-cfe31ee0abaf'},
  'RetryAttempts': 0},
 'modelSummaries': [{'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-tg1-large',
   'modelId': 'amazon.titan-tg1-large',
   'modelName': 'Titan Text Large',
   'providerName': 'Amazon',
   'inputModalities': ['TEXT'],
   'outputModalities': ['TEXT'],
   'responseStreamingSupported': True,
   'customizationsSupported': [],
   'inferenceTypesSupported': ['ON_DEMAND'],
   'modelLifecycle': {'status': 'ACTIVE'}},
  {'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-image-generator-v1:0',
   'modelId': 'amazon.titan-image-generator-v1:0',
   'modelName': 'Titan Image Generator G1',
   'providerName': 'Amazon',

In [10]:
# Use the native inference API to send a text message to Amazon Titan Text G1 - Express.

import boto3
import json

from botocore.exceptions import ClientError

# Create an Amazon Bedrock Runtime client.
brt = boto3.client("bedrock-runtime")

# Set the model ID, e.g., Amazon Titan Text G1 - Express.
# model_id = "amazon.titan-text-express-v1"
# model_id = "anthropic.claude-3-5-haiku-20241022-v1:0"
model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"


# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Format the request payload using the model's native structure.
# native_request = {
#     "inputText": prompt,
#     "textGenerationConfig": {
#         "maxTokenCount": 512,
#         "temperature": 0.5,
#         "topP": 0.9
#     },
# }

native_request = {
  "modelId": "anthropic.claude-3-5-haiku-20241022-v1:0",
  "contentType": "application/json",
  "accept": "application/json",
  "body": {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 200,
    "top_k": 250,
    "stopSequences": [],
    "temperature": 1,
    "top_p": 0.999,
    "messages": [
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": "hello world"
          }
        ]
      }
    ]
  }
}

# Convert the native request to JSON.
request = json.dumps(native_request)

try:
    # Invoke the model with the request.
    response = brt.invoke_model(modelId=model_id, body=request)

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
    exit(1)

# Decode the response body.
model_response = json.loads(response["body"].read())

# Extract and print the response text.
response_text = model_response["results"][0]["outputText"]
print(response_text)

ERROR: Can't invoke 'us.anthropic.claude-3-5-sonnet-20241022-v2:0'. Reason: An error occurred (ValidationException) when calling the InvokeModel operation: Malformed input request: #: required key [prompt] not found#: required key [max_tokens_to_sample] not found#: extraneous key [modelId] is not permitted#: extraneous key [body] is not permitted#: extraneous key [contentType] is not permitted#: extraneous key [accept] is not permitted, please reformat your input and try again.


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [1]:
def generate_question_evaluation_prompt(question):
    prompt = f"""
As an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder's knowledge, which may require implicit or external information. Your goal is to analyze the question based on the following criteria:

- **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)
- **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)
- **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question's effectiveness or specificity. (True if multiple answers are plausible; False if only one valid answer is expected.)
    
#### Output JSON Keys:
- `question`: The input question.
- `flags`: 
    - `logical_structure`: (True/False)
    - `redundancy`: (True/False)
    - `multiple_answers`: (True/False)
    
- `reasoning`: 
    - `logical_structure`: Reason for the logical structure flag.
    - `redundancy`: Reason for the redundancy flag.
    - `multiple_answers`: Reason for the multiple answers flag.

#### Task:
Analyze the following question and provide a JSON object containing flags and reasons for potential issues:

**Question**: "{question}"

#### Output:
Return a JSON object that evaluates the question based on the criteria above.
"""
    return prompt

In [ ]:
def generate_question_evaluation_prompt(question, answer, triplets):
    prompt = f"""
    As an expert evaluator, your role is to:
    (1) assess the quality and validity of trivia or natural questions. 
    (2) check if the provided answer addresses the informational need of the question.
    (3) check if the provided supporting triplets sufficiently support or justify the answer.
    These questions aim to test the responder's knowledge, which may require implicit or external information. 
    
    
    For step (1), your goal is to analyze the question based on the following criteria:

    - **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)
    - **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)
    - **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question's effectiveness or specificity. (True if multiple answers are plausible; False if only one valid answer is expected.)
    
    For step (2), your goal is to analyze:
    - **Informational Sufficiency**:  if the answer meets the information need  of the question (True if the answer is sufficient; False if the answer is insufficient.)

    For step (3), your goal is to analyze:
    - **Supporting Triplets**: if the supporting triplets sufficiently support or justify the answer. (True if the triplets support the answer; False if the triplets do not support the answer.)

    #### Output JSON Keys:
    - `question`: The input question.
    - `flags`: 
      - `logical_structure`: (True/False)
      - `redundancy`: (True/False)
      - `multiple_answers`: (True/False)
      - `informational_sufficiency`: (True/False)
      - `supporting_triplets`: (True/False)
      
    - `reasoning`: 
      - `logical_structure`: Reason for the logical structure flag.
      - `redundancy`: Reason for the redundancy flag.
      - `multiple_answers`: Reason for the multiple answers flag.
      - `informational_sufficiency`: Reason for the informational sufficiency flag.
      - `supporting_triplets`: Reason for the supporting triplets flag.

    #### Task:
    Analyze the following question, answer, and triplets,  and provide a JSON object containing flags and reasons for potential issues:

    **Question**: "{question}"
    **Answer**: "{answer}"
    **Triplets**: {triplets}

    #### Output:
    Return a JSON object that evaluates the question based on the criteria above.
    """
    return prompt